In [1]:
import json
import pandas as pd
from pathlib import Path
from prompt import build_messages

ROOT = Path.cwd().parent
SRC = ROOT/"Dataset"/"issues_train.csv"
OUT = ROOT/"Dataset"/"lora"
OUT.mkdir(exist_ok=True)

df = pd.read_csv(SRC)
df = df.fillna("")

valid = df.groupby("label").sample(n=50, random_state=42)
train = df.drop(valid.index)
train = train.sample(frac=1, random_state=42)   # shuffle, the file is sorted in blocks of 100
print(len(train), len(valid))
print(train["label"].value_counts())
print(valid["label"].value_counts())

1350 150
label
question    450
bug         450
feature     450
Name: count, dtype: int64
label
bug         50
feature     50
question    50
Name: count, dtype: int64


In [2]:
def write_jsonl(part, path):
    with open(path, "w", encoding="utf-8") as f:
        for i, row in part.iterrows():
            case = {
                "title": row["title"].strip(),
                "body": row["body"].replace("\r\n", "\n").strip()[:1500]
            }

            messages = build_messages(case)
            messages.append({"role": "assistant", "content": row["label"]})   # the answer the model should learn
            f.write(json.dumps({"messages": messages}, ensure_ascii=False) + "\n")

write_jsonl(train, OUT/"train.jsonl")
write_jsonl(valid, OUT/"valid.jsonl")